In [20]:
from imutils import paths
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from random import shuffle
import random
import timeit

from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.utils import class_weight

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import AveragePooling2D, Dropout, Flatten, Dense, Input, BatchNormalization
from tensorflow.keras.models import Model, model_from_json
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import losses
from tensorflow.keras import datasets, layers, models
import tensorflow as tf

import tensorflow_addons as tfa

from tensorflow.keras.applications import imagenet_utils
from tensorflow.keras.applications.xception import decode_predictions

In [2]:
from pyFile import ARModel
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
import numpy as np

import matplotlib.pyplot as plt

In [3]:
modelAvl = ["vgg16","vgg19","inception","xception","resnet50","resnet101","densenet","inceptionResnet"]
modelSel = ["inception", "xception" ]
modelOnly = ["xception"]
losses = ["bce", "cce", "focal", "kld"]
lossSel = ["bce", "focal"]
lossOnly = ["focal"]

In [4]:
def fix_gpu():
    config = ConfigProto()
    config.gpu_options.allow_growth = True
    session = InteractiveSession(config=config)

fix_gpu()

2022-10-28 11:29:02.558047: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-10-28 11:29:02.560995: I tensorflow/compiler/jit/xla_gpu_device.cc:99] Not creating XLA devices, tf_xla_enable_xla_devices not set
2022-10-28 11:29:02.568414: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2022-10-28 11:29:02.613819: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:941] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-10-28 11:29:02.613975: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:01:00.0 na

In [5]:
model = ARModel()

In [6]:
(data, labels) = model.loadImages(r'/home/bishal/Research/Allergic-Rhinitis/Dataset/all/rotate', 
        plotType="R", classification="multiclass", colorMode="RGB", cleanImageF=True, resize=True,
        correctColor=False, contours=False, crop=True ,printImgDemo=False)

[INFO]: Trying to Read the images from  /home/bishal/Research/Allergic-Rhinitis/Dataset/all/rotate
Images found : 90


In [7]:
(data, labels) = model.prepareData(data, labels, weightedLossCalc=True)

[INFO]: Preparing Data
{'imgNumber': 111, 'dataInfo': 'R', 'classification': 'multiclass', 'imageCount': 90, 'imageType': 'R', 'classType': 'multiclass', 'colorMode': 'RGB', 'clean': True, 'crop': True, 'resize': True, 'colorCorrect': False, 'imgDim': (224, 224, 3)}


In [8]:
trainAug = model.setDataAugmentation(normalizeData=False, rotate=2, zoom=0.15, wShift=0.2, hShift=0.2, 
                                shear=0.15, hFlip=True, vFlip=False, generateImages=False)


[INFO]: Augmenting Data with - 
{'rotate': 2, 'zoom': 0.15, 'wShift': 0.2, 'hShift': 0.2, 'shear': 0.15, 'hFlip': True, 'vFlip': False}


In [9]:
#plt.imshow(trainX[54])

In [10]:
#(trainX, trainY, testX, testY) = model.setPartition(data, labels, testSize=0.20)

In [11]:
#testY = np.expand_dims(testY, 0)
#testX = np.expand_dims(testX, 0)

In [13]:
#testY

In [15]:
#testX.shape

In [16]:
currBModel = model.setBaseModel("xception")
                
currHModel = model.setHeadModel(currBModel, dropoutRate=0.5, activation="siren")
finalModel = model.initModel(currBModel, currHModel, baseTrainable=False)
model.setHyperParameters(learningRate = 1e-3, epochs = 200, batchSize = 8)
finalModel = model.compileModel(finalModel, loss="focal")

2022-10-28 11:29:37.435097: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2022-10-28 11:29:37.435321: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:941] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-10-28 11:29:37.435528: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA TITAN RTX computeCapability: 7.5
coreClock: 1.77GHz coreCount: 72 deviceMemorySize: 23.65GiB deviceMemoryBandwidth: 625.94GiB/s
2022-10-28 11:29:37.435585: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.10.1
2022-10-28 11:29:37.435610: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.10
2022-10-28 11:29:37.435628: I tensorflow/stream_executor/platform/defaul

[INFO]: Model Selected -  xception
[INFO]: Initializing Model
[INFO]: Hyperparameters Set
[INFO]: Compiling Model


In [ ]:
y_hat = []
y = []
start = timeit.default_timer()
for i in range(len(data)):
    print("#### Iter - %s ####"%str(i+1))
    trainX = np.delete(data, [i], axis=0)
    trainY = np.delete(labels, [i], axis=0)
    testX = np.expand_dims(data[i], 0)
    testY = np.expand_dims(labels[i], 0)
    y.append(labels[i])
    print("Sizes : ", trainX.shape, "--", trainY.shape)
    print("Sizes : ", testX.shape, "--", testY.shape)
    
    (H, finalModel) = model.startTraining(finalModel, trainAug, trainX, trainY, 
                                            testX, testY, weightedLoss=True, learningDecay=False, earlyStop=True)
    
    predIdxs = model.startTesting(testX, testY, finalModel, voting=0)
    y_hat.append(predIdxs[0])
stop = timeit.default_timer()

#### Iter - 1 ####
Sizes :  (89, 224, 224, 3) -- (89, 3)
Sizes :  (1, 224, 224, 3) -- (1, 3)
[INFO] Model Training
Epoch 1/200
 1/11 [=>............................] - ETA: 0s - loss: 0.1934 - accuracy: 0.6250

In [ ]:
print("Total Time - ",stop-start)

In [ ]:
len(y)

In [ ]:
y_org = []
for val in y:
    if val[0]:
        y_org.append(0)
    elif val[1]:
        y_org.append(1)
    elif val[2]:
        y_org.append(2)

In [ ]:
y_org= y_org[1:]
len(y_org)

In [ ]:
count = 0
for i in range(len(y_org)):
     if y_org[i] == y_hat[i]:
            count += 1
acc = int((count / len(y_org) ) * 100)
acc

In [ ]:
y_ = np.array(y)
y_hat_ = np.array(y_hat)

In [ ]:
(H, finalModel) = model.startTraining(finalModel, trainAug, trainX, trainY, 
                                                    testX, testY, weightedLoss=True, learningDecay=False, earlyStop=True)

In [ ]:
predIdxs = model.startTesting(testX, testY, finalModel, voting=30)

In [ ]:
labels[i]

In [ ]:
predIdxs

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
cm= confusion_matrix(y_.argmax(axis=1), y_hat_)
print(cm)
total = sum(sum(cm))
acc = (cm[0,0] + cm[1,1] + cm[2,2]) / total
sensitivity = cm[0, 0] / (cm[0, 0] + cm[0, 1] + cm[0,2])
specificity = cm[1, 1] / (cm[1, 0] + cm[1, 1] + cm[1,2])
specificity2 = cm[2, 2] / (cm[2, 0] + cm[2, 1] + cm[2,2])
specificity = (specificity + specificity2) / 2

In [ ]:
testY.shape

In [ ]:
preds = np.expand_dims(predIdxs, 0)

In [ ]:
print(classification_report(y_[1:].argmax(axis=1), y_hat_, target_names=['0', '1', '2']))

In [ ]:
model.evalModel(predIdxs, testY)